# 梯度下降法：從手算到框架

對應補充教材《深度學習 2023_0601_R2》第 30 至 39 頁。

投影片上的更新式是

$$\theta^{(k+1)} = \theta^{(k)} - \eta \nabla L(\theta^{(k)})$$

其中 $L$ 是損失函數 loss function，$\theta$ 是參數 parameters，
$\eta$ 是學習率 learning rate，$\nabla L$ 是 $L$ 對各參數的偏微分組成的梯度向量。

這份筆記分五段：

| 段 | 做什麼 |
|---|---|
| 1 | 一維：手算一次，程式再跑一次，兩邊數字必須一樣 |
| 2 | 學習率太小、剛好、太大，各跑一次看差別 |
| 3 | 二維：等高線與法線方向，看懂投影片第 31 頁的那張圖 |
| 4 | 框架版：TensorFlow 2 的 GradientTape 與 SGD，數字要和手寫版一致 |
| 5 | 對照投影片第 32 頁的那三個數字 |

In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

## 1. 一維：先手算，再讓程式算

用一個算得出來的例子，這樣才驗證得了程式有沒有寫對。

$$L(w) = (w-3)^2 \qquad \frac{dL}{dw} = 2(w-3)$$

起點 $w^{(0)} = 0$，學習率 $\eta = 0.1$。最小值顯然在 $w = 3$。

**先自己算第一步**：$\frac{dL}{dw}\Big|_{w=0} = 2(0-3) = -6$，
所以 $w^{(1)} = 0 - 0.1 \times (-6) = 0.6$。

梯度是負的，所以 $w$ 往**正**的方向移動——往下坡走，不是往梯度的方向走。

In [ ]:
def loss(w):
    """L(w) = (w - 3)^2"""
    return (w - 3.0) ** 2


def grad(w):
    """dL/dw = 2(w - 3)，這一行就是整個「微分」，手寫版要自己推出來"""
    return 2.0 * (w - 3.0)


def gradient_descent_1d(w0=0.0, eta=0.1, steps=6, verbose=True):
    """最原始的梯度下降：算梯度、往反方向走一小步、重複。"""
    w, history = w0, []
    if verbose:
        print('%-6s %-12s %-12s %-12s' % ('step', 'w', 'dL/dw', 'L(w)'))
        print('-' * 46)
    for k in range(steps + 1):
        g = grad(w)
        history.append((k, w, g, loss(w)))
        if verbose:
            print('%-6d %-12.6f %-12.6f %-12.6f' % (k, w, g, loss(w)))
        w = w - eta * g            # ← 更新式就是這一行
    return np.array(history)


hist = gradient_descent_1d()

第 1 步算出 0.600000，和手算的 0.6 一樣。

這個例子有閉式解可以反查：每一步 $w-3$ 都會乘上 $(1-2\eta)$，
所以 $w^{(k)} = 3 - 3(1-2\eta)^k$。$\eta = 0.1$ 時係數是 0.8，
也就是每一步距離終點都只剩上一步的八成。

In [ ]:
ETA = 0.1
print('%-6s %-14s %-14s %s' % ('step', '程式迭代', '閉式解', '差'))
for k, w, g, L in hist:
    closed = 3 - 3 * (1 - 2 * ETA) ** k
    print('%-6d %-14.6f %-14.6f %.1e' % (k, w, closed, abs(w - closed)))

## 2. 學習率的影響

投影片第 37 頁說學習率會影響結果，但沒給數字。這裡三種各跑一次。

收斂係數是 $(1-2\eta)$：

| $\eta$ | $1-2\eta$ | 會怎樣 |
|---|---|---|
| 0.01 | 0.98 | 每步只縮 2%，走很久還到不了 |
| 0.1 | 0.80 | 穩定收斂 |
| 1.05 | −1.10 | 絕對值大於 1，每步都放大並換邊，發散 |

判準很乾淨：$|1-2\eta| < 1$ 才會收斂，也就是 $0 < \eta < 1$。

In [ ]:
for eta in (0.01, 0.1, 1.05):
    h = gradient_descent_1d(eta=eta, steps=5, verbose=False)
    print('eta=%-5s 係數 %-6.2f  w: %s'
          % (eta, 1 - 2 * eta, '  '.join('%8.4f' % v for v in h[:, 1])))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
xs = np.linspace(-2, 8, 400)
for ax, eta in zip(axes, (0.01, 0.1, 1.05)):
    h = gradient_descent_1d(eta=eta, steps=5, verbose=False)
    ax.plot(xs, loss(xs), color='#1F4E79', lw=1.5)
    ax.plot(h[:, 1], h[:, 3], 'o-', color='#C03030', ms=5, lw=1.2)
    for k, w, g, L in h:
        ax.annotate('%d' % k, (w, L), textcoords='offset points',
                    xytext=(4, 5), fontsize=8, color='#C03030')
    ax.axvline(3, ls='--', color='grey', lw=1)
    ax.set_title('eta = %s   factor = %.2f' % (eta, 1 - 2 * eta))
    ax.set_xlabel('w'); ax.set_ylabel('L(w)')
    ax.set_ylim(-1, 30)
plt.tight_layout()
plt.show()

左圖走不動，中圖穩定下降，右圖左右跳而且越跳越遠。

## 3. 二維：等高線與法線方向

投影片第 31 頁寫「Gradient: Loss 的等高線的法線方向」。用一個兩個參數的函數來看：

$$L(\theta_1, \theta_2) = \theta_1^2 + 10\,\theta_2^2
\qquad \nabla L = (2\theta_1,\; 20\theta_2)$$

$\theta_2$ 的係數是 $\theta_1$ 的十倍，等高線被壓成一個狹長的橢圓。
起點 $(3, 1)$，$\eta = 0.08$。

這一步的梯度是 $(6, 20)$——兩個分量差超過三倍，所以兩個方向的更新幅度完全不同：

* $\theta_1$ 每步乘 $1-0.08\times 2 = 0.84$，慢慢靠近
* $\theta_2$ 每步乘 $1-0.08\times 20 = -0.60$，每一步都跨過去再跳回來

In [ ]:
def loss2(t):
    return t[0] ** 2 + 10 * t[1] ** 2


def grad2(t):
    return np.array([2 * t[0], 20 * t[1]])


def gradient_descent_2d(t0=(3.0, 1.0), eta=0.08, steps=5):
    t, history = np.array(t0, float), []
    print('%-6s %-10s %-10s %-10s %-10s %s'
          % ('step', 't1', 't2', 'dL/dt1', 'dL/dt2', 'L'))
    print('-' * 60)
    for k in range(steps + 1):
        g = grad2(t)
        history.append((t[0], t[1]))
        print('%-6d %-10.4f %-10.4f %-10.4f %-10.4f %.4f'
              % (k, t[0], t[1], g[0], g[1], loss2(t)))
        t = t - eta * g
    return np.array(history)


path = gradient_descent_2d()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
g1, g2 = np.meshgrid(np.linspace(-0.5, 3.6, 300), np.linspace(-1.2, 1.2, 300))
cs = ax.contour(g1, g2, g1 ** 2 + 10 * g2 ** 2, levels=[0.5, 2, 5, 10, 19, 32],
                colors='#1F4E79', linewidths=1)
ax.clabel(cs, inline=True, fontsize=7)
ax.plot(path[:, 0], path[:, 1], 'o-', color='#C03030', ms=5, lw=1.4)
for k, (a, b) in enumerate(path):
    ax.annotate('%d' % k, (a, b), textcoords='offset points',
                xytext=(5, 5), fontsize=9, color='#C03030')
# 起點的梯度向量：方向與等高線垂直，而移動是它的反方向
g0 = grad2(path[0])
ax.arrow(path[0][0], path[0][1], g0[0] * 0.03, g0[1] * 0.03,
         head_width=0.05, color='#1E7A45', lw=1.8, length_includes_head=True)
ax.annotate('gradient at step 0\n(perpendicular to contour)',
            (path[0][0] + 0.18, path[0][1] + 0.6), fontsize=8, color='#1E7A45')
ax.plot(0, 0, 'k*', ms=12)
ax.set_xlabel('theta_1'); ax.set_ylabel('theta_2')
ax.set_title('Zig-zag caused by unequal feature scales')
plt.tight_layout()
plt.show()

這張鋸齒圖就是小考第 1 次考過的那題：各特徵尺度差距大時，
梯度下降會在陡峭方向劇烈震盪、在平坦方向幾乎不動。
解法不是把學習率調小，調小了平坦方向會更走不動；
正確做法是把特徵先標準化，讓等高線變回接近圓形。

## 4. 框架版：TensorFlow 2

手寫版要自己推導 `dL/dw = 2(w-3)`。框架版不必推導，
`GradientTape` 會把前向計算錄下來，再自動反向算出梯度。

兩版跑同一題，數字必須一樣——這是驗證框架有沒有被誤用的方法。

In [ ]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf

print('TensorFlow', tf.__version__)

w_tf = tf.Variable(0.0)
# legacy.SGD 在 TF 2.13 的 CPU 上與課本行為一致；新版 optimizer 介面不同
opt = tf.keras.optimizers.legacy.SGD(learning_rate=0.1)

print('%-6s %-12s %-12s %-12s' % ('step', 'w', 'dL/dw', 'L(w)'))
print('-' * 46)
tf_hist = []
for k in range(7):
    with tf.GradientTape() as tape:
        L = (w_tf - 3.0) ** 2          # 只要寫前向計算
    g = tape.gradient(L, w_tf)         # 反向自動微分
    tf_hist.append((k, float(w_tf), float(g), float(L)))
    print('%-6d %-12.6f %-12.6f %-12.6f' % (k, float(w_tf), float(g), float(L)))
    opt.apply_gradients([(g, w_tf)])   # 等同 w = w - eta * g

In [ ]:
worst = max(max(abs(a[1] - b[1]), abs(a[2] - b[2]))
            for a, b in zip(hist, tf_hist))
print('手寫版與框架版的最大差 %.2e' % worst)
print('判定：%s' % ('一致' if worst < 1e-5 else '不一致，要查是哪一邊寫錯'))

差距是 $10^{-7}$ 等級，來自單精度浮點數，不是演算法不同。

框架版少寫的那一行是微分。模型一複雜，手推微分就不可行了，
這是框架真正的價值；但也因此，**框架算錯你不會收到任何錯誤訊息**，
只會得到一個不收斂的模型。能用一個手算得出答案的小題目去驗證，是必要的能力。

## 5. 對照補充教材第 32 頁

投影片第 32 頁給了三個參數 0.2、−0.1、0.3，和它們算出來的梯度 0.15、0.05、0.2，
但沒寫更新後是多少。補上：

In [ ]:
theta = np.array([0.2, -0.1, 0.3])
gradient = np.array([0.15, 0.05, 0.2])
for eta in (0.1,):
    print('theta      =', theta)
    print('gradient   =', gradient)
    print('eta        =', eta)
    print('theta_new  =', theta - eta * gradient)

三個參數同時更新，每一個都是「自己減掉學習率乘自己的偏微分」，彼此不互相影響。
投影片上一整排的 `Compute` 就是在做這件事，只是參數從 3 個變成幾百萬個。

---

## 練習

1. 把 `gradient_descent_1d` 的 `eta` 改成 0.5，先用閉式解預測會發生什麼，再跑跑看。
2. 把 `loss2` 改成 $\theta_1^2 + \theta_2^2$，等高線會變成什麼形狀？鋸齒還在嗎？
3. 把框架版的 `legacy.SGD` 換成 `legacy.Adam`，數字還會和手寫版一樣嗎？為什麼？

In [ ]:
if __name__ == '__main__':
    print('\n全部跑完。')